<div style="
background: linear-gradient(135deg, #f8f9fa 0%, #edf6f9 45%, #e8eaf6 100%);
padding: 40px;
border-radius: 20px;
text-align: center;
font-family: 'Segoe UI', sans-serif;
box-shadow: 0 8px 24px rgba(0,0,0,0.08);
border: 1px solid #dce3ea;
">

  <h1 style="
  color: #5c6b8a;
  font-size: 2.2em;
  margin: 0 0 8px 0;
  letter-spacing: 1px;
  font-weight: 700;">
  🤖 CP020003 — Artificial Intelligence 2026
  </h1>

  <h2 style="
  color: #7b8fa1;
  font-size: 1.3em;
  margin: 0 0 16px 0;
  font-weight: 400;">
  Khon Kaen University
  </h2>

  <hr style="
  border: 1px solid #c9d6df;
  width: 60%;
  margin: 18px auto;">

  <p style="color: #495057; font-size: 1.05em; margin: 6px 0;">
    👨‍🏫 <strong style="color:#6c7aa1;">Author:</strong>
    Teerapong Panboonyuen (P'Kao)
  </p>

  <p style="color: #495057; font-size: 1.05em; margin: 6px 0;">
    📧 <strong style="color:#6c7aa1;">Contact:</strong>
    teerapong.pa@chula.ac.th
  </p>

  <p style="color: #495057; font-size: 1.05em; margin: 6px 0;">
    🏫 <strong style="color:#6c7aa1;">Course:</strong>
    AI 2026 @ KKU
  </p>

  <p style="color: #495057; font-size: 1.05em; margin: 6px 0;">
    📦 <strong style="color:#6c7aa1;">GitHub:</strong>
    <a href="https://github.com/kaopanboonyuen/CP020003_ArtificialIntelligence_2026s1"
       style="color:#5b8def; text-decoration:none;">
       CP020003_ArtificialIntelligence_2026s1
    </a>
  </p>

  <hr style="
  border: 1px solid #c9d6df;
  width: 60%;
  margin: 18px auto;">

  <p style="
color: #6c757d;
font-size: 0.95em;
margin: 4px 0;">
📚 Built with inspiration from the open-source AI community:
<strong style="color:#7286a0;">
Python · Pandas · NumPy · scikit-learn · PyTorch · Hugging Face · Kaggle
</strong>
</p>

  <p style="
  color: #8a97a6;
  font-size: 0.9em;
  margin-top: 12px;
  font-style: italic;">
  "This notebook is open to everyone — including those who cannot afford university.
  Knowledge is for all. 🌏"
  </p>

</div>

## 🧬 DNA Classification: From Random Forests to LSTMs (PyTorch Edition)
### CP020003 Artificial Intelligence — In-Class Notebook (Leveled-Up Version 💪)

We are living in the age of **precision healthcare**, where doctors and researchers believe that changes in
**DNA** — tiny mutations — can drive the emergence of new diseases, including new **virus variants** that
mutate faster than we can track them by hand. This is exactly the kind of pattern-recognition problem that
Machine Learning and Deep Learning were built for.

Over the last 3 weeks you learned: basic Python + pandas, feature engineering, and supervised learning with
both **scikit-learn** and **TensorFlow**. This week we look at a completely different kind of data — raw
**DNA sequences** — and we bring in a new deep learning toolkit: **PyTorch** 🔥.

By the end of this notebook you will be able to:
1. Explain, in plain computer-science terms, what DNA, a genome, and a mutation actually are
2. Load a real dataset from GitHub and clean up junk files (`.DS_Store`, `__MACOSX`) safely
3. Explore a genomics dataset (EDA) and honestly check whether a signal even exists before modeling
4. Engineer real, biology-inspired features — including a **k-mer frequency vector**, the "bag-of-words"
   of genomics
5. Train and compare 3 classic ML models: **Decision Tree**, **Random Forest**, **XGBoost**
6. Build 3 deep learning models **from scratch in PyTorch**: a motif-detecting **1D CNN**, a **bidirectional
   LSTM**, and a **bidirectional GRU** — all trained directly on the raw A/C/G/T sequence
7. Understand, layer by layer, *why* each architecture is built the way it is
8. Fairly compare **6 models total** (ML + DL) against a random-guess baseline
9. Save and reload the winning model's weights, then run inference on a brand-new DNA sequence

Runs fully on **Google Colab (free CPU or GPU)** — every model in this notebook trains in well under a
minute. If you have extra time in class, we'll point out exactly where you can bump up the epochs for a
bit more accuracy. ⏱️


## 0. Setup

We need pandas/numpy/matplotlib/seaborn for data work, scikit-learn + xgboost for classic ML, and
**PyTorch** for deep learning. Colab already ships with PyTorch pre-installed, so this cell is fast.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from itertools import product

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier

from sklearn.metrics import (
    accuracy_score, f1_score, confusion_matrix, ConfusionMatrixDisplay,
    classification_report
)

sns.set_theme(style="whitegrid")
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
torch.manual_seed(RANDOM_STATE)

# Use a GPU if Colab gives us one (Runtime -> Change runtime type -> GPU), otherwise CPU is totally fine here
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Libraries loaded ✅")
print("Using device:", device)


## 1. A 5-Minute Biology Crash Course (for Computer Scientists) 🧫

You don't need a biology degree for this notebook — just a few definitions, explained the way a programmer
would explain them:

| Biology term | What it actually is | CS analogy |
|---|---|---|
| **Nucleotide** | One of 4 building blocks: **A** (Adenine), **C** (Cytosine), **G** (Guanine), **T** (Thymine) | One character in a 4-letter alphabet `{A, C, G, T}` |
| **DNA sequence** | A long string made of those 4 letters, e.g. `"ACGGTTAC..."` | A `string` variable |
| **Genome** | The *entire* DNA sequence of an organism (billions of letters for humans!) | The full source code / codebase of an organism |
| **Gene** | A meaningful sub-sequence that codes for something specific (like a protein) | A `function` inside the codebase |
| **Mutation** | A change in the DNA sequence — a letter flips, gets deleted, or inserted | A code change / commit — could be a harmless typo, or it could introduce a serious bug |
| **Motif** | A short, recurring pattern in DNA that has a specific biological meaning (e.g. "start reading here") | A recognizable syntax pattern in code, like `def ...():` always marking a function |
| **GC Content** | The % of the sequence that is `G` or `C` (as opposed to `A` or `T`) | A simple summary statistic, like "% of your code that is comments" |
| **k-mer** | A sub-string of length *k* taken from the sequence (e.g. all 3-letter chunks: `"ACG"`, `"CGG"`, ...) | Exactly the same idea as an **n-gram** in NLP! |

### Why does this matter for healthcare? 🏥

Every virus, bacterium, plant, and human cell carries its own DNA. When DNA **mutates**, it can change how an
organism behaves — sometimes harmlessly, sometimes enough to create a **new disease variant** (think of how
often you hear about a new COVID "variant" in the news — that's a virus whose genome picked up mutations).

Real genomes contain **motifs** — short, recognizable sequence patterns that mark specific biological
functions. For example:
- `GAATTC` is the real recognition site cut by the **EcoRI** restriction enzyme, widely used in bacterial
  genetics and biotech labs
- `TATAAA` is the real **TATA box**, a promoter motif found just upstream of many human genes
- `CCAAT` is the real **CAAT box**, another common eukaryotic promoter element

Being able to look at a DNA sequence and spot patterns like these — or predict what organism it likely came
from — is a real task computational biologists and ML engineers work on every day. Today, *you* are that ML
engineer.


### 1.1 Bonus Biology: Chromosomes, Proteins, and the "Central Dogma" 🧬📦

A few more pieces of vocabulary that will make you sound like you know what you're talking about at your next
biotech meetup:

| Term | What it is | CS analogy |
|---|---|---|
| **Chromosome** | A tightly-packed bundle of DNA. Humans have 46 chromosomes (23 pairs) that together hold the entire genome | Splitting one giant codebase into multiple repository files/modules — same total code, organized into chunks |
| **Protein** | A molecule built by reading a gene and translating it, 3 letters ("codon") at a time, into a chain of amino acids | The *compiled output* your source code produces when it actually runs |
| **Central Dogma** | The classic biology pipeline: **DNA → RNA → Protein** — DNA is "transcribed" into RNA, then RNA is "translated" into protein | Source code → intermediate representation (bytecode) → the running executable |

Putting the whole picture together:

```
Genome  (the entire codebase)
  └── Chromosome  (one file/module in that codebase)
        └── Gene  (one function inside that file)
              └── DNA sequence  (the literal source code text: A, C, G, T)
                    └── transcribed + translated into →  Protein  (the compiled, running program)
```

Mutations matter precisely because a single-letter change deep in a gene can change the resulting protein —
sometimes with zero effect (like renaming a private variable), and sometimes breaking something critical
(like flipping a `>` to a `<` in a condition). That's the whole story behind why tracking DNA changes is such
a big deal in modern healthcare, and why the classification task you're about to build is a genuinely useful
skill in computational biology.

> 🧠 **Quick check for the class:** if a *genome* is the whole codebase and a *chromosome* is one file, what
> would you call a single line of that file? (Answer: pretty close to a single **nucleotide** — one `A`,
> `C`, `G`, or `T`!)


## 2. Get the Data (from GitHub) 📦

The dataset is a synthetic (safely made-up) collection of **3,000 DNA sequences**, 100 letters each, labelled
with:
- `Class_Label`: which kind of organism the sequence likely came from — **Bacteria / Human / Plant / Virus**
- `Disease_Risk`: **Low / Medium / High**
- Plus pre-computed features like `GC_Content`, `kmer_3_freq`, and a `Mutation_Flag`

We'll download the zip, unzip it, and make sure we **ignore junk files** like `.DS_Store` (a macOS system
file) and the `__MACOSX` folder that macOS sometimes sneaks into zip archives.


In [ ]:
# ==========================
# TODO: Student: Enter your dataset name
# ==========================
YOUR_ZIP_FILE_NAME = ''

url = "https://github.com/kaopanboonyuen/CP020003_ArtificialIntelligence_2026s1/raw/main/dataset/" + YOUR_ZIP_FILE_NAME + ".zip"
output = YOUR_ZIP_FILE_NAME + ".zip"

!wget -q "{url}" -O "{output}"

In [ ]:
# Unzip using Python's zipfile, skipping .DS_Store and __MACOSX junk that macOS sometimes adds to zips
import zipfile, os

extract_dir = "dna-synthetic"
with zipfile.ZipFile("dna-synthetic.zip", "r") as zf:
    for member in zf.namelist():
        if ".DS_Store" in member or "__MACOSX" in member:
            continue  # skip junk files
        zf.extract(member, extract_dir)

print("Extracted files:")
for root, _, files in os.walk(extract_dir):
    for f in files:
        print(" -", os.path.join(root, f))

In [ ]:
csv_path = # Write your dataset path here

df = pd.read_csv(csv_path)
print(df.shape)
df.head()

In [ ]:
df.info()

In [ ]:
# Any missing values?
# Write your code here

## 3. Exploratory Data Analysis (EDA) — and an Honest Sanity Check 🔍

Before touching any model, let's *get to know* the data. Three quick questions:
1. Are the classes balanced?
2. Do GC content / k-mer frequency look different across organism types?
3. Is there any obvious relationship between `Mutation_Flag` and `Disease_Risk`?


In [ ]:
# 1. Class balance
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
df["Class_Label"].value_counts().plot(kind="bar", ax=axes[0], color="teal")
axes[0].set_title("Class_Label distribution")
axes[0].set_ylabel("count")

df["Disease_Risk"].value_counts().plot(kind="bar", ax=axes[1], color="indianred")
axes[1].set_title("Disease_Risk distribution")
plt.tight_layout()
plt.show()


In [ ]:
# 2. Do GC_Content and kmer_3_freq differ across organism classes?
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
sns.boxplot(x="Class_Label", y="GC_Content", data=df, ax=axes[0])
axes[0].set_title("GC_Content by Class_Label")

sns.boxplot(x="Class_Label", y="kmer_3_freq", data=df, ax=axes[1])
axes[1].set_title("kmer_3_freq by Class_Label")
plt.tight_layout()
plt.show()


In [ ]:
# 3. Mutation_Flag vs Disease_Risk
pd.crosstab(df["Mutation_Flag"], df["Disease_Risk"], normalize="index").round(3)


📝 **Honest finding:** notice the boxplots above look *very* similar across classes, and the crosstab
barely changes with `Mutation_Flag`. If you trained models directly on these numbers right now, every model —
Decision Tree, Random Forest, XGBoost, even a CNN — would land close to a random-guess baseline (~25%
accuracy for 4 balanced classes). We actually tried this, and confirmed it isn't a bug: these particular
provided columns simply don't carry enough class-specific signal in this synthetic sample.

**That is a completely normal, real thing that happens in ML** — but it's not a satisfying place to end a
lesson on model comparison! So in Section 4 we're going to do what a real bioinformatician would do: go
looking for actual sequence **motifs** instead of relying only on pre-computed summary statistics, and
level up our feature representation so our models have something real to learn. 💪


## 4. Feature Engineering — 2 New, Biology-Inspired Features 🧪

First, two quick features real genomics researchers actually use:

| # | New Feature | Formula | Why it helps |
|---|---|---|---|
| 1 | `Purine_Pyrimidine_Ratio` | `(Num_A + Num_G) / (Num_T + Num_C)` | `A` and `G` are chemically classified as **purines**; `T` and `C` are **pyrimidines**. This ratio captures a real structural property of the DNA strand. |
| 2 | `GC_Skew` | `(Num_G - Num_C) / (Num_G + Num_C)` | **GC skew** is a classic genomics statistic that measures asymmetry between G and C — it's used in real research to help locate the origin of DNA replication! |


In [ ]:
eps = 1e-6  # tiny number to avoid division by zero

df["Purine_Pyrimidine_Ratio"] = # Write your code here
df["GC_Skew"] = (df["Num_G"] - # Write your code here

df[["Purine_Pyrimidine_Ratio", "GC_Skew"]].describe()


## 4.5 Leveling Up — Injecting Realistic Biological Motifs 💪🧬

Real DNA classification works because real organisms carry **distinctive short motifs** — like the
`GAATTC`, `TATAAA`, and `CCAAT` motifs mentioned in Section 1. Our synthetic dataset didn't originally bake
any organism-specific motif into the sequences (that's *why* Section 3 found no signal).

To give you a genuinely satisfying "the model actually works!" experience, we'll simulate what a real,
motif-carrying dataset looks like: we assign **one short marker motif per class** and stamp it into ~85% of
that class's sequences, at a **random position** each time (never in exactly the same spot — this makes it a
fair, realistic detection task instead of a trivial lookup):

| Class | Motif | Real biological meaning |
|---|---|---|
| Bacteria | `GAATTC` | EcoRI restriction-enzyme recognition site |
| Human | `TATAAA` | TATA box (promoter element) |
| Plant | `CCAAT` | CAAT box (promoter element) |
| Virus | `TTGACA` | Viral-promoter-style motif |

> 💡 **Full transparency for the class:** this is a deliberately *enriched* copy of the sequence
> (`Sequence_v2`), built on top of the original data, so that our model-comparison exercise has real signal to
> find — exactly what you'd want a genomics ML pipeline to detect in practice. We are not changing the
> original `Sequence` column or hiding this step; it's 100% visible in the code below.


In [ ]:
motifs = {
    # Write your code here
}

def inject_motif(seq, motif, p=0.85):
    '''Stamp `motif` into `seq` at a random position, ~p of the time (some sequences stay motif-free, on purpose, to keep the task realistic).'''
    seq = list(seq)
    if np.random.rand() < p:
        pos = np.random.randint(0, len(seq) - len(motif))
        seq[pos:pos + len(motif)] = list(motif)
    return "".join(seq)

df["Sequence_v2"] = [inject_motif(s, motifs[c]) for s, c in zip(df["Sequence"], df["Class_Label"])]

# Sanity check: does each motif actually show up mostly in its OWN class, and rarely in the others?
for cls, m in motifs.items():
    within = df.loc[df["Class_Label"] == cls, "Sequence_v2"].str.contains(m).mean()
    cross = df.loc[df["Class_Label"] != cls, "Sequence_v2"].str.contains(m).mean()
    print(f"{cls:10s} motif={m:8s}  found-in-own-class={within:.2f}   found-in-other-classes={cross:.2f}")


## 5. Prepare the Data for Modeling 🎯

**Target:** `Class_Label` — given a 100-letter DNA sequence, can a model guess whether it came from
**Bacteria, Human, Plant, or Virus**? This is a **4-class classification problem**.

For the **classical ML models** (Section 6), instead of just a single `kmer_3_freq` average, we'll build a
full **4-mer frequency vector**: count every possible 4-letter combination (`4^4 = 256` of them) inside each
sequence and turn that into 256 numeric features. This is exactly the "bag-of-words" idea from NLP, applied
to DNA — and it's rich enough to actually capture our injected motifs.

For the **deep learning models** (Section 8), we go one step further and feed the **raw sequence itself** —
no hand-crafted features at all, the network learns its own.

We always split data **before** scaling, and use a **stratified split** so all 4 classes stay proportionally
represented in both train and test sets.


In [ ]:
K = 4  # k-mer length
ALL_KMERS = ["".join(p) for p in product("ACGT", repeat=K)]
print(f"Using {len(ALL_KMERS)} distinct {K}-mers as features (e.g. {ALL_KMERS[:5]} ...)")

def kmer_frequency_vector(seq, k=K, all_kmers=ALL_KMERS):
    '''Turns a DNA string into a normalized frequency vector over every possible k-mer.'''
    counts = dict.fromkeys(all_kmers, 0)
    for i in range(len(seq) - k + 1):
        sub = seq[i:i + k]
        counts[sub] += 1
    total = len(seq) - k + 1
    return np.array([counts[km] / total for km in all_kmers], dtype=np.float32)

kmer_features = np.stack([kmer_frequency_vector(s) for s in df["Sequence_v2"]])
print("k-mer feature matrix shape:", kmer_features.shape)


In [ ]:
label_encoder = LabelEncoder()
df["Class_Label_enc"] = label_encoder.fit_transform(df["Class_Label"])
print("Classes:", list(label_encoder.classes_), "-> encoded as", list(range(len(label_encoder.classes_))))

X = # Write your code here
y = # Write your code here
sequences = df["Sequence_v2"].values

X_train, X_test, y_train, y_test, seq_train, seq_test = train_test_split(
    X, y, sequences, test_size=0.2, random_state=RANDOM_STATE, stratify=y
)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("Train size:", X_train.shape, " Test size:", X_test.shape)


> 🧹 **Pro tips (เคล็ดลับแม่บ้าน) — 3 small habits that save you from big headaches:**
> 1. **`stratify=y`** in `train_test_split` — without it, a random split could accidentally put almost all of
>    one class into the test set, wrecking your metrics for reasons that have nothing to do with your model
> 2. **Fit the scaler on training data only** (`scaler.fit_transform(X_train)`, then just
>    `scaler.transform(X_test)`) — fitting on the *whole* dataset leaks information from the test set into
>    training, which quietly inflates your accuracy
> 3. **Always set `random_state`** — every time you re-run this notebook, you should get the *same* split,
>    the *same* results, and be able to compare fairly with a teammate who ran the exact same code


In [ ]:
# Our honesty check for later: what accuracy would a model get by ALWAYS guessing the majority class?
majority_class_share = pd.Series(y_train).value_counts(normalize=True).max()
print(f"Majority-class baseline accuracy: {majority_class_share:.3f}")


## 6. Classical Machine Learning Models 🌳

We'll train 3 tree-based models, from simplest to most powerful, on our 256-dimensional k-mer feature table.

### 6.1 Decision Tree — "20 Questions" for data
A Decision Tree asks a series of yes/no questions about the features (e.g. *"is the frequency of `GAAT` >
0.01?"*), each question splitting the data into two branches, until it reaches a leaf that predicts a class.
It's easy to visualize and interpret, but a single deep tree can **overfit** — it can memorize noise in the
training data.

### 6.2 Random Forest — "ask a crowd of trees, take a vote"
A Random Forest trains **many** Decision Trees, each one on a random subset of the data *and* a random subset
of features, then lets all the trees **vote** on the final prediction. This "wisdom of the crowd" approach
usually generalizes much better than any single tree, and with 256 k-mer features to choose from, different
trees can each latch onto a piece of the motif signal.

### 6.3 XGBoost — "trees that learn from each other's mistakes"
XGBoost (e**X**treme **G**radient **Boost**ing) also builds many trees, but **sequentially**: each new tree is
trained specifically to fix the errors the previous trees made. This "boosting" strategy is one of the most
consistently high-performing methods on structured/tabular data, and is a staple in ML competitions.

```python
# We are commenting out a classic shallow neural network (MLPClassifier) here on purpose —
# for structured tabular features like ours, tree-based models are usually just as good or better,
# and MUCH faster to train. We'll save "real" deep learning for Section 8, where we let a neural
# network read the raw DNA sequence directly instead of hand-crafted numbers.
#
# from sklearn.neural_network import MLPClassifier
# mlp = MLPClassifier(hidden_layer_sizes=(64, 32), max_iter=300, random_state=RANDOM_STATE)
# mlp.fit(X_train_scaled, y_train)
```


In [ ]:
results = {}  # we'll keep collecting every model's test performance here

def evaluate_and_store(name, y_true, y_pred, y_proba=None):
    acc = accuracy_score(y_true, y_pred)
    f1_macro = f1_score(y_true, y_pred, average="macro")
    results[name] = {"accuracy": acc, "f1_macro": f1_macro, "y_pred": y_pred}
    print(f"{name:20s}  accuracy={acc:.3f}   f1_macro={f1_macro:.3f}")
    return acc, f1_macro


### 6.1 Decision Tree — Key Hyperparameters 🎛️

| Parameter | What it controls | How to tune it |
|---|---|---|
| `max_depth` | The maximum number of yes/no questions the tree can ask before it must make a prediction | **Lower** (e.g. 3-5) → simpler tree, less overfitting, but might underfit. **Higher** (e.g. 15+, or `None` for unlimited) → the tree can memorize the training data (overfit). Start small and increase while watching test accuracy. |
| `min_samples_split` | The minimum number of samples a node must have before it's allowed to split further | **Higher** values (e.g. 20-50) force the tree to stop splitting on tiny, noisy groups — a simple, effective anti-overfitting knob |
| `min_samples_leaf` | The minimum number of samples allowed in a final leaf | Similar idea to above — prevents the tree from creating a leaf for just 1-2 training examples (classic overfitting sign) |
| `criterion` | The math formula used to decide which question is "best" at each split — `"gini"` or `"entropy"` | Both usually give similar results; `"entropy"` is slightly more expensive to compute but sometimes marginally more accurate |
| `random_state` | The random seed | Always set this! It makes your results **reproducible** — same code, same data, same answer every time |

> 🧹 **Pro tip (เคล็ดลับแม่บ้าน):** if your Decision Tree gets much higher accuracy on the training set than
> on the test set, that's the classic sign of overfitting — the fix is almost always to **lower `max_depth`**
> or **raise `min_samples_leaf`**, not to add more data tricks.


In [ ]:
# 6.1 Decision Tree
dt_model = # Write your code here
dt_model.fit(X_train_scaled, y_train)
y_pred_dt = dt_model.predict(X_test_scaled)
evaluate_and_store("Decision Tree", y_test, y_pred_dt)


### 6.2 Random Forest — Key Hyperparameters 🎛️

| Parameter | What it controls | How to tune it |
|---|---|---|
| `n_estimators` | How many individual trees are in the "forest" | **More trees** (e.g. 100 → 500) usually means steadier, slightly better accuracy — but training takes longer. Past a certain point (often ~200-500) returns diminish, so it's rarely worth going much higher just for accuracy |
| `max_depth` | Max depth of each individual tree (same idea as the Decision Tree above) | Random Forests are naturally more overfit-resistant than a single tree (because of averaging), so you can usually afford a **deeper** `max_depth` than you would for a lone Decision Tree |
| `max_features` | How many features each tree is allowed to consider at each split (e.g. `"sqrt"`, `"log2"`, or a number) | Restricting this **forces trees to be more different from each other**, which is exactly what makes "wisdom of the crowd" work — too high and all your trees start looking the same |
| `min_samples_leaf` | Same as Decision Tree — minimum samples per leaf | Same anti-overfitting logic as above |
| `n_jobs` | How many CPU cores to use for training in parallel | Set to `-1` to use every core available — free speed, no accuracy trade-off! |
| `random_state` | The random seed | Set this for reproducibility, same as always |

> 🧹 **Pro tip:** a Random Forest's accuracy tends to plateau — try `n_estimators=100`, then `300`, then
> `600`, and plot the test accuracy for each. Once the curve flattens, you've found the sweet spot and can
> stop adding trees.


In [ ]:
# 6.2 Random Forest
rf_model = # Write your code here
rf_model.fit(X_train_scaled, y_train)
y_pred_rf = rf_model.predict(X_test_scaled)
evaluate_and_store("Random Forest", y_test, y_pred_rf)


### 6.3 XGBoost — Key Hyperparameters 🎛️

| Parameter | What it controls | How to tune it |
|---|---|---|
| `n_estimators` | How many boosting rounds (trees added sequentially) | More rounds can keep improving accuracy, but **too many** with a high `learning_rate` will start overfitting — this is why `n_estimators` and `learning_rate` are usually tuned *together* |
| `max_depth` | Depth of each individual boosted tree | XGBoost trees are usually kept **shallow** (3-6) on purpose — boosting works by combining many *weak* learners, not a few strong ones |
| `learning_rate` (a.k.a. `eta`) | How much each new tree is allowed to correct the previous trees' mistakes | **Lower** (e.g. 0.01-0.05) → more careful, usually more accurate, but needs **more** `n_estimators` to compensate and takes longer to train. **Higher** (e.g. 0.2-0.3) → faster training, higher risk of overfitting |
| `subsample` | Fraction of training rows randomly sampled for each boosting round | Values like 0.7-0.9 add helpful randomness that reduces overfitting (similar spirit to Random Forest's `max_features`) |
| `colsample_bytree` | Fraction of features randomly sampled for each tree | Same idea as `subsample`, but for columns instead of rows |
| `eval_metric` | The loss function XGBoost optimizes and reports during training | `"mlogloss"` (multi-class log-loss) is the standard choice for multi-class classification like ours |

> 🧹 **Pro tip:** the classic XGBoost tuning recipe is **"lower the learning rate, raise the number of
> trees"** — e.g. going from `learning_rate=0.1, n_estimators=100` to `learning_rate=0.02, n_estimators=500`
> often squeezes out a bit more accuracy, at the cost of longer training time. Great homework experiment!


In [ ]:
# 6.3 XGBoost
xgb_model = XGBClassifier(
    # Write your code here
)
xgb_model.fit(X_train_scaled, y_train)
y_pred_xgb = xgb_model.predict(X_test_scaled)
evaluate_and_store("XGBoost", y_test, y_pred_xgb)


## 7. Evaluate the Classical Models — and Check Against a Baseline ⚖️

An accuracy number means nothing on its own — we always compare against a **baseline**. The simplest
possible baseline is: *"always predict the most common class"*. Let's see how far past that our upgraded
k-mer features got us.


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, (name, pred) in zip(axes, [("Decision Tree", y_pred_dt), ("Random Forest", y_pred_rf), ("XGBoost", y_pred_xgb)]):
    cm = confusion_matrix(y_test, pred)
    disp = ConfusionMatrixDisplay(cm, display_labels=label_encoder.classes_)
    disp.plot(ax=ax, colorbar=False, xticks_rotation=45)
    ax.set_title(name)
plt.tight_layout()
plt.show()


In [ ]:
print(f"Majority-class baseline accuracy: {majority_class_share:.3f}\n")
for name in ["Decision Tree", "Random Forest", "XGBoost"]:
    print(f"{name:15s} accuracy = {results[name]['accuracy']:.3f}")


📝 **Discussion for class:** the diagonal of each confusion matrix should now look much stronger than in
Section 3's plain-feature version — that jump is coming entirely from richer **feature representation**
(256 k-mer frequencies instead of one averaged number) plus the injected motifs actually being present to
find. This is a great real-world lesson: sometimes the fix for a "the model doesn't work" problem isn't a
fancier algorithm — it's giving the model a **feature representation that can actually see the signal**.


## 8. Now Let's Go Deep — PyTorch on the *Raw* DNA Sequence 🔥

So far even our upgraded classical models only saw a k-mer frequency table. Now we let a neural network read
the **actual 100-letter sequence** and decide for itself what patterns matter — this is the whole promise of
deep learning: **automatic feature learning**, no manual k-mer counting required.

### 8.1 Turning a DNA string into numbers a network can understand

Neural networks only understand numbers, so we need to **encode** each sequence. We'll use two encodings:

- **One-hot encoding** (for our CNN): each letter becomes a length-4 vector, e.g. `A -> [1,0,0,0]`,
  `C -> [0,1,0,0]`, `G -> [0,0,1,0]`, `T -> [0,0,0,1]`. A 100-letter sequence becomes a `4 x 100` matrix —
  exactly like a tiny "image" with 4 channels instead of the usual 3 (RGB)!
- **Integer encoding** (for our LSTM/GRU): each letter becomes a single integer, e.g. `A=0, C=1, G=2, T=3`.
  We then feed these integers into an **Embedding layer**, which is how recurrent networks usually consume
  sequences (just like word embeddings in NLP!).


In [ ]:
nuc2idx = # Write your code here
SEQ_LEN = len(seq_train[0])
NUM_CLASSES = len(label_encoder.classes_)
print("Sequence length:", SEQ_LEN, " | Number of classes:", NUM_CLASSES)

def one_hot_encode(seq):
    '''Turns 'ACGT...' into a (4, seq_len) float32 matrix.'''
    arr = np.zeros((4, len(seq)), dtype=np.float32)
    for i, ch in enumerate(seq):
        arr[nuc2idx[ch], i] = 1.0
    return arr

def integer_encode(seq):
    '''Turns 'ACGT...' into a list of integers, e.g. [0, 1, 2, 3, ...].'''
    return [nuc2idx[ch] for ch in seq]

# Quick sanity check
print(seq_train[0][:20], "...")
print(one_hot_encode(seq_train[0])[:, :5])
print(integer_encode(seq_train[0])[:10])


In [ ]:
class DNASequenceDatasetOneHot(Dataset):
    '''PyTorch Dataset for the CNN: returns a (4, seq_len) one-hot tensor + integer label.'''
    def __init__(self, seqs, labels):
        self.seqs = seqs
        self.labels = labels

    def __len__(self):
        return len(self.seqs)

    def __getitem__(self, idx):
        x = one_hot_encode(self.seqs[idx])
        y = self.labels[idx]
        return torch.tensor(x), torch.tensor(y, dtype=torch.long)


class DNASequenceDatasetInt(Dataset):
    '''PyTorch Dataset for the LSTM/GRU: returns a (seq_len,) integer tensor + integer label.'''
    def __init__(self, seqs, labels):
        self.seqs = seqs
        self.labels = labels

    def __len__(self):
        return len(self.seqs)

    def __getitem__(self, idx):
        x = integer_encode(self.seqs[idx])
        y = self.labels[idx]
        return torch.tensor(x, dtype=torch.long), torch.tensor(y, dtype=torch.long)


BATCH_SIZE = # Write your code here

onehot_train_ds = DNASequenceDatasetOneHot(seq_train, y_train)
onehot_test_ds = DNASequenceDatasetOneHot(seq_test, y_test)
onehot_train_dl = DataLoader(onehot_train_ds, batch_size=BATCH_SIZE, shuffle=True)
onehot_test_dl = DataLoader(onehot_test_ds, batch_size=BATCH_SIZE)

int_train_ds = DNASequenceDatasetInt(seq_train, y_train)
int_test_ds = DNASequenceDatasetInt(seq_test, y_test)
int_train_dl = DataLoader(int_train_ds, batch_size=BATCH_SIZE, shuffle=True)
int_test_dl = DataLoader(int_test_ds, batch_size=BATCH_SIZE)

print("Batches per epoch (train):", len(onehot_train_dl))


> 🧹 **Pro tip — `shuffle=True` on the train loader, never on the test loader:** shuffling the *training*
> data every epoch stops the model from accidentally learning the order sequences appear in. We deliberately
> leave `shuffle=False` (the default) on the *test* loader — order doesn't matter there, and keeping it fixed
> makes debugging easier (row 0 of your predictions always lines up with row 0 of your test set).


In [ ]:
def train_pytorch_model(model, train_dl, test_dl, epochs=15, lr=2e-3, model_name="model"):
    '''Generic training loop we'll reuse for CNN, LSTM, and GRU. Tracks loss + accuracy every epoch so we can plot learning curves afterwards.'''
    model = model.to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    criterion = nn.CrossEntropyLoss()

    history = {"train_loss": [], "test_accuracy": []}

    for epoch in range(epochs):
        model.train()
        total_loss = 0.0
        for xb, yb in train_dl:
            xb, yb = xb.to(device), yb.to(device)
            optimizer.zero_grad()
            logits = model(xb)
            loss = criterion(logits, yb)
            loss.backward()
            optimizer.step()
            total_loss += loss.item()
        avg_loss = total_loss / len(train_dl)

        # Check test accuracy at the end of every epoch (cheap for a dataset this size, and great for plotting)
        model.eval()
        correct, total = 0, 0
        with torch.no_grad():
            for xb, yb in test_dl:
                xb, yb = xb.to(device), yb.to(device)
                preds = model(xb).argmax(dim=1)
                correct += (preds == yb).sum().item()
                total += yb.size(0)
        test_acc = correct / total

        history["train_loss"].append(avg_loss)
        history["test_accuracy"].append(test_acc)

        if (epoch + 1) % 3 == 0 or epoch == epochs - 1:
            print(f"  [{model_name}] epoch {epoch+1}/{epochs}  train_loss={avg_loss:.4f}  test_acc={test_acc:.3f}")

    # Final predictions on the test set (for the confusion matrix / leaderboard later)
    model.eval()
    all_preds, all_targets = [], []
    with torch.no_grad():
        for xb, yb in test_dl:
            xb = xb.to(device)
            logits = model(xb)
            preds = logits.argmax(dim=1).cpu().numpy()
            all_preds.extend(preds)
            all_targets.extend(yb.numpy())
    return model, np.array(all_targets), np.array(all_preds), history


def plot_training_curves(history, model_name):
    '''Plots the classic "impress the crowd" pair: training loss going down, test accuracy going up.'''
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))

    axes[0].plot(history["train_loss"], color="crimson", marker="o", markersize=3)
    axes[0].set_title(f"{model_name}: Training Loss per Epoch")
    axes[0].set_xlabel("Epoch")
    axes[0].set_ylabel("Loss")

    axes[1].plot(history["test_accuracy"], color="seagreen", marker="o", markersize=3)
    axes[1].set_title(f"{model_name}: Test Accuracy per Epoch")
    axes[1].set_xlabel("Epoch")
    axes[1].set_ylabel("Accuracy")
    axes[1].set_ylim(0, 1)

    plt.tight_layout()
    plt.show()


> 🧹 **Pro tip — reading a learning curve like a pro:** a healthy training run shows **loss steadily
> dropping** and **accuracy steadily climbing**, both flattening out toward the end. Watch out for these two
> warning signs:
> - **Loss stops dropping almost immediately** → your `lr` (learning rate) is probably too small, or the
>   model is too simple for the task
> - **Loss goes down but jumps around wildly / spikes upward** → your `lr` is probably too large; try cutting
>   it in half (e.g. `2e-3 -> 1e-3`)
> - **Accuracy plateaus early while loss keeps dropping** → classic overfitting sign; more `epochs` won't
>   help, but more `Dropout` or fewer parameters might


### 8.2 CNN — a Motif-Detecting 1D Convolutional Network 🧱

A **Conv1D** layer slides a small window (a "kernel") along the sequence, learning to detect short local
patterns — similar to how a CNN on images detects edges or textures, but here it's scanning for short DNA
**motifs** like the ones we injected in Section 4.5.

The key design choice that makes this a genuine **motif detector** (this exact recipe is used in real
genomics deep learning models like DeepBind) is the **global max pool**: instead of flattening every
position into the classifier, we ask "what is the *single strongest match* this filter found, anywhere in
the sequence?" That makes the model **position-invariant** — it doesn't matter whether the motif shows up at
position 12 or position 84, the filter will still fire.

```
Input (4 channels x 100 positions, one-hot encoded)
   -> Conv1d(4 -> 32 filters, kernel=8)  + ReLU     # each filter learns to detect one candidate motif shape
   -> Global Max Pool over the whole sequence        # "did THIS filter's motif appear anywhere at all?"
   -> Linear -> ReLU -> Dropout                      # a small classifier head
   -> Linear -> 4 output classes
```

Each of the 32 filters is a tiny pattern-detector, `8` letters wide, that slides across the whole sequence
and reports its single strongest activation — exactly what we need to find a specific motif regardless of
where it landed.

### 🏗️ How to Design Your Own Deep Learning Architecture (a general checklist)

Every architecture in this notebook was built by answering the same 5 questions, in order — use this
checklist whenever you're designing a new network from scratch, not just for DNA:

1. **What shape is my input?** → decides your first layer. A sequence of categories (like `A/C/G/T`) usually
   starts with an `Embedding` (for RNNs) or a one-hot + `Conv1d` (for CNNs). An image starts with `Conv2d`. A
   flat table of numbers starts with `Linear`.
2. **Does position matter, or just presence?** → "does this pattern exist *anywhere* in the input?" calls for
   pooling (max or average) across the whole sequence, exactly like our `AdaptiveMaxPool1d` / `outputs.max()`
   calls. "What happened at the *end*?" (like predicting the next word) calls for using the final hidden
   state instead.
3. **How much capacity do I actually need?** → start **small** (few filters/hidden units, 1-2 layers). Only
   add more `num_filters`, `hidden_dim`, or layers if the model is *underfitting* (both train and test
   accuracy are low). Adding capacity to an already-overfitting model just makes things worse.
4. **What's my output shape?** → decides your last layer. Classification into `N` classes → `Linear(..., N)`
   with no activation (softmax is applied inside `CrossEntropyLoss` automatically). Predicting a number →
   `Linear(..., 1)`.
5. **Where might it overfit, and how do I guard against it?** → add `Dropout` before your final classifier
   layers, and keep an eye on the loss/accuracy curves (see the pro tip above) to catch it early.

| Hyperparameter | What it controls | Typical starting point |
|---|---|---|
| `num_filters` (CNN) / `hidden_dim` (RNN) | How much "capacity" each layer has to represent patterns | Start small (16-32); double it only if you see underfitting |
| `kernel_size` (CNN) | How many letters wide each motif-detector filter is | Should roughly match your real motif length — we used `8` since our motifs are 5-6 letters long |
| `embed_dim` (RNN) | How many numbers represent each letter after the Embedding layer | 4-16 is plenty for a 4-symbol alphabet like DNA (compare: word embeddings for English often use 100-300!) |
| `dropout` | Fraction of neurons randomly "switched off" during training, to prevent overfitting | 0.2-0.5 is a common, safe range |
| `lr` (learning rate) | How big a step the optimizer takes after each batch | `1e-3` to `3e-3` with the Adam optimizer is a reliable default for small networks like these |
| `epochs` | How many full passes through the training data | Watch the accuracy curve — stop increasing once it flattens out |
| `batch_size` | How many sequences are processed together per optimizer step | 32-128 is typical; bigger batches train faster per-epoch on a GPU but need a bit more memory |


In [ ]:
class DNA_MotifCNN(nn.Module):
    def __init__(self, num_classes=NUM_CLASSES, num_filters=32, kernel_size=8):
        super().__init__()
        self.conv = nn.Conv1d(in_channels=4, out_channels=num_filters, kernel_size=kernel_size)
        self.relu = nn.ReLU()
        self.global_pool = nn.AdaptiveMaxPool1d(1)  # take the single strongest match anywhere in the sequence
        self.dropout = nn.Dropout(0.3)
        self.fc1 = nn.Linear(num_filters, 32)
        self.fc2 = nn.Linear(32, num_classes)

    def forward(self, x):
        # x shape: (batch, 4, seq_len)
        x = self.relu(self.conv(x))              # -> (batch, num_filters, seq_len - kernel_size + 1)
        x = self.global_pool(x).squeeze(-1)       # -> (batch, num_filters)  <- position-invariant!
        x = self.dropout(self.relu(self.fc1(x)))  # -> (batch, 32)
        return self.fc2(x)                        # -> (batch, num_classes)

torch.manual_seed(RANDOM_STATE)  # reset seed so each model gets a consistent, reproducible initialization
cnn_model = DNA_MotifCNN()
print(cnn_model)


In [ ]:
# Trains in just a few seconds per epoch on CPU, even faster on GPU
torch.manual_seed(RANDOM_STATE)  # reset again so the DataLoader shuffling is reproducible too
cnn_model, y_test_cnn, y_pred_cnn, cnn_history = train_pytorch_model(
    cnn_model, onehot_train_dl, onehot_test_dl, epochs=15, model_name="CNN"
)
evaluate_and_store("CNN (PyTorch)", y_test_cnn, y_pred_cnn)


In [ ]:
plot_training_curves(# Write your code here)


> ⏱️ **If you have extra time in class:** try raising `epochs` to 25-30, adding a 2nd Conv1d layer, or
> increasing `num_filters` to 64 for a bit more accuracy.

### 8.3 & 8.4 LSTM and GRU — Reading the Sequence Step by Step 🧠⚡

An **LSTM** reads the sequence **one letter at a time**, left to right, keeping a running "memory" (the
**hidden state**) that it updates at every step, using 3 internal **gates**:

- **Forget gate** — decides what old information to throw away from memory
- **Input gate** — decides what new information to add to memory
- **Output gate** — decides what to output based on the current memory

A **GRU** is a lighter cousin of the LSTM: just **2 gates** (**update** and **reset**) and a single hidden
state instead of two, making it a bit faster to train while often reaching similar accuracy.

**Two upgrades vs. a "plain" recurrent classifier**, both important for motif-finding tasks like ours:
1. **Bidirectional** — we run the sequence both left-to-right *and* right-to-left, and combine both views.
   This means the network never has to "remember" something from 90 steps ago; a motif near the end of the
   sequence is just as easy to catch as one near the start.
2. **Max-pool over every timestep**, not just the final one — a plain RNN classifier usually only looks at
   the hidden state *after the very last letter*, which forces it to carry every important detail all the
   way to the end. Instead, we let the model take the strongest signal from *any* position — exactly the
   same "does this pattern appear anywhere?" idea we used for the CNN's global max pool.

```
Input (integer-encoded sequence, length 100)
   -> Embedding(4 letters -> 8-dim vectors)
   -> Bidirectional LSTM / GRU (hidden_size=32, both directions -> 64-dim per position)
   -> Max-pool over all 100 positions                 # "did this pattern show up anywhere?"
   -> Linear -> ReLU -> Dropout
   -> Linear -> 4 output classes
```

| Layer | What it does | Output shape |
|---|---|---|
| `nn.Embedding(4, 8)` | Turns each integer (0-3) into a learnable 8-dimensional vector — the network gets to decide what makes `A` "similar to" or "different from" `G` | `(batch, 100, 8)` |
| `nn.LSTM` / `nn.GRU` (`bidirectional=True`) | Reads the embedded sequence left-to-right AND right-to-left, producing one combined vector per position | `(batch, 100, 64)` |
| `.max(dim=1)` | Collapses the 100 per-position vectors down to 1, by taking the strongest value in each of the 64 dimensions | `(batch, 64)` |
| `nn.Linear(64, 32)` + `ReLU` + `Dropout` | A small classifier "head" that combines the pooled signal | `(batch, 32)` |
| `nn.Linear(32, 4)` | The final output layer — one raw score per class | `(batch, 4)` |


In [ ]:
class DNA_RNN(nn.Module):
    '''Shared implementation for both the LSTM and GRU variants (just swap `cell_type`).'''
    def __init__(self, cell_type="lstm", vocab_size=4, embed_dim=8, hidden_dim=32, num_classes=NUM_CLASSES):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim)
        rnn_cls = nn.LSTM if cell_type == "lstm" else nn.GRU
        self.rnn = rnn_cls(input_size=embed_dim, hidden_size=hidden_dim, batch_first=True, bidirectional=True)
        self.cell_type = cell_type
        self.dropout = nn.Dropout(0.3)
        self.relu = nn.ReLU()
        self.fc1 = nn.Linear(hidden_dim * 2, 32)  # x2 because bidirectional concatenates both directions
        self.fc2 = nn.Linear(32, num_classes)

    def forward(self, x):
        # x shape: (batch, seq_len) of integers 0-3
        x = self.embedding(x)                 # -> (batch, seq_len, embed_dim)
        outputs, _ = self.rnn(x)              # -> (batch, seq_len, hidden_dim * 2), one vector PER position
        pooled, _ = outputs.max(dim=1)        # -> (batch, hidden_dim * 2)  <- strongest signal at any position
        x = self.dropout(self.relu(self.fc1(pooled)))
        return self.fc2(x)

torch.manual_seed(RANDOM_STATE)
lstm_model = DNA_RNN(cell_type="lstm")
print(lstm_model)


In [ ]:
torch.manual_seed(RANDOM_STATE)
lstm_model, y_test_lstm, y_pred_lstm, lstm_history = train_pytorch_model(
    lstm_model, int_train_dl, int_test_dl, epochs=20, model_name="LSTM"
)
evaluate_and_store("LSTM (PyTorch)", y_test_lstm, y_pred_lstm)


In [ ]:
plot_training_curves(# Write your code here)


In [ ]:
torch.manual_seed(RANDOM_STATE)
gru_model = DNA_RNN(cell_type="gru")
print(gru_model)


In [ ]:
torch.manual_seed(RANDOM_STATE)
gru_model, y_test_gru, y_pred_gru, gru_history = train_pytorch_model(
    gru_model, int_train_dl, int_test_dl, epochs=20, model_name="GRU"
)
evaluate_and_store("GRU (PyTorch)", y_test_gru, y_pred_gru)


In [ ]:
plot_training_curves(# Write your code here)


## 9. Final Leaderboard — Classical ML vs Deep Learning 🏆

Let's line up all 6 models side by side, plus our majority-class baseline, and see who actually wins.


In [ ]:
summary = pd.DataFrame({
    name: {"accuracy": r["accuracy"], "f1_macro": r["f1_macro"]}
    for name, r in results.items()
}).T.sort_values("accuracy", ascending=False)

summary.loc["Majority-class baseline"] = [majority_class_share, np.nan]
summary = summary.sort_values("accuracy", ascending=False)
summary.round(3)


In [ ]:
plt.figure(figsize=(8, 5))
plot_df = summary.drop("Majority-class baseline").sort_values("accuracy")
plt.barh(plot_df.index, plot_df["accuracy"], color="steelblue")
plt.axvline(majority_class_share, color="red", linestyle="--", label="Majority-class baseline")
plt.xlabel("Test accuracy")
plt.title("Model Comparison: Classical ML vs Deep Learning")
plt.legend()
plt.tight_layout()
plt.show()

best_model_name = # Write your code here
print("Best model:", best_model_name)


📝 **Discussion for class:** every model should now be clearing the majority-class baseline by a wide
margin — because this time, there is genuine signal (our injected motifs) for them to find. Notice *how* each
model got there:
- The tree-based models needed a **richer feature representation** (256 k-mer frequencies) to expose the
  signal at all
- The CNN and RNNs needed an **architecture choice** (global/max pooling across the whole sequence) that
  matches the nature of the task: "does this pattern appear *anywhere*?", not "what's the very last letter?"

That's the real lesson of this notebook: a fancier model is never automatically better — what matters is
whether your **features** and your **architecture** actually match the structure of the problem you're
trying to solve.


## 10. Save & Load the Winning Model, Then Try Inference 💾🔍

A trained model is useless if you can't reuse it later. Let's save the winning PyTorch model's weights to
disk, reload them into a fresh model instance (proving the save/load actually works), and then run
**inference** on a brand new DNA sequence.


In [ ]:
# We'll demonstrate save/load with the CNN model (swap in lstm_model or gru_model if one of those won!)
torch.save(cnn_model.state_dict(), "dna_cnn_weights.pt")
print("Saved weights to dna_cnn_weights.pt")

# Load into a FRESH model instance to prove it works
loaded_cnn = DNA_MotifCNN()
loaded_cnn.load_state_dict(torch.load("dna_cnn_weights.pt", map_location=device))
loaded_cnn.to(device)
loaded_cnn.eval()
print("Weights loaded into a fresh model ✅")


In [ ]:
def predict_sequence(model, seq, encoding="onehot"):
    '''Run inference on a single raw DNA string (must be same length as training sequences).'''
    model.eval()
    with torch.no_grad():
        if encoding == "onehot":
            x = torch.tensor(one_hot_encode(seq)).unsqueeze(0).to(device)  # add batch dimension
        else:
            x = torch.tensor(integer_encode(seq), dtype=torch.long).unsqueeze(0).to(device)
        logits = model(x)
        probs = torch.softmax(logits, dim=1).cpu().numpy()[0]
        pred_idx = probs.argmax()
    return label_encoder.classes_[pred_idx], probs

# Try it on a real test-set sequence
sample_seq = seq_test[0]
predicted_class, probs = predict_sequence(loaded_cnn, sample_seq, encoding="onehot")

print("Sequence: ", sample_seq)
print("Predicted class:", predicted_class)
for cls, p in zip(label_encoder.classes_, probs):
    print(f"  {cls:10s}: {p:.3f}")


In [ ]:
# Bonus: build our OWN test sequence with the Bacteria motif stamped in, and see if the model catches it
import random
random.seed(1)
random_background = "".join(random.choice("ACGT") for _ in range(SEQ_LEN))
test_bacteria_seq = list(random_background)
test_bacteria_seq[40:46] = list("GAATTC")  # stamp in the EcoRI / Bacteria motif
test_bacteria_seq = "".join(test_bacteria_seq)

predicted_class, probs = predict_sequence(loaded_cnn, test_bacteria_seq, encoding="onehot")
print("Custom sequence with GAATTC stamped in:", test_bacteria_seq)
print("Predicted class:", predicted_class)
for cls, p in zip(label_encoder.classes_, probs):
    print(f"  {cls:10s}: {p:.3f}")


> 🎓 **Try it yourself:** change the stamped-in motif above to `TATAAA`, `CCAAT`, or `TTGACA` and see if
> the model correctly switches its prediction to Human, Plant, or Virus!


## 11. Wrap-Up — What We Learned Today 🎉

1. **DNA is just a string** over a 4-letter alphabet, and a genome is the entire "codebase" of an organism
   (Section 1)
2. Always **exclude junk files** (`.DS_Store`, `__MACOSX`) when unzipping real-world data (Section 2)
3. **EDA first** — and don't be afraid to discover that a dataset has *no* signal; that's a valid, useful
   finding (Section 3)
4. Real biological **motifs** (like `GAATTC`, `TATAAA`, `CCAAT`) are short, recurring patterns with real
   biological meaning — and richer **k-mer feature representations** are what let classical ML models see
   them (Sections 4, 4.5, 6)
5. We trained **3 classical ML models** — Decision Tree, Random Forest, XGBoost — each building on the idea
   of the last (Section 6)
6. We trained **3 deep learning models in PyTorch**, straight from the raw sequence:
   - **CNN**: scans for local motifs with sliding convolution filters + **global max pooling**
   - **Bidirectional LSTM**: reads the sequence both forward and backward with gated memory
   - **Bidirectional GRU**: a lighter, faster cousin of the LSTM
   - Both RNNs use **max-pooling over every timestep**, not just the last one — the same "does this pattern
     appear anywhere?" trick as the CNN
   (Section 8)
7. **Always compare against a baseline** (majority-class guessing) before trusting an accuracy number
   (Sections 7 & 9)
8. The real win here wasn't a fancier algorithm — it was matching **features and architecture** to the
   actual structure of the problem
9. A trained model is only useful once you can **save its weights and reload them** for real inference
   (Section 10)

### Final leaderboard (recap)


In [ ]:
summary.round(3)

---

<div style="
background: linear-gradient(135deg, #fafafa 0%, #eef6f9 50%, #e8eaf6 100%);
padding: 30px;
border-radius: 18px;
text-align: center;
font-family: 'Segoe UI', sans-serif;
box-shadow: 0 6px 18px rgba(0,0,0,0.06);
border: 1px solid #dce3ea;
">

  <h2 style="
  color: #5c6b8a;
  margin: 0 0 12px 0;
  font-size: 1.8em;
  font-weight: 700;">
  🎉 Well Done!
  </h2>

  <p style="
  color: #495057;
  font-size: 1.05em;
  margin: 6px 0;">
  You've completed the Week 4 Notebook for
  <strong style="color:#6c7aa1;">
  CP020003 — AI 2026 @ KKU
  </strong>
  </p>

  <!--
  <p style="
  color: #6c757d;
  font-size: 0.95em;
  margin-top: 12px;">
  Next week we dive into
  <strong style="color:#5b8def;">
  Supervised Learning
  </strong>
  — scikit-learn, train/test splits, and your first ML model 🚀
  </p>
  -->

  <hr style="
  border: 1px solid #c9d6df;
  width: 50%;
  margin: 16px auto;">

  <p style="
  color: #7d8790;
  font-size: 0.9em;
  font-style: italic;
  margin-bottom: 6px;">
  "Shared freely so that everyone, everywhere, can learn AI."
  </p>

  <p style="
  color: #8a97a6;
  font-size: 0.85em;">
  — Teerapong Panboonyuen (P'Kao) · teerapong.pa@chula.ac.th
  </p>

</div>